# D5 · From a VP metadata record to a notebook

This notebook starts from one thing only, the IRI of a dataset record in the ERDERA Virtual Platform, and follows the metadata to the data. Every URL it touches comes out of the record. Nothing else is typed in.

The chain is portal row → dataset record (FAIR Data Point) → distribution record → data file → first analysis → provenance record.

Today the portal's resource list shows these datasets without a link to their record, so the IRI below has to be taken from the VP Index or from the FDP that published the dataset (here the EJP RD WP13 FDP, `https://w3id.org/ejp-rd/fairdatapoints/wp13`). That gap is part of what this demonstrator shows.

In [ ]:
# Parameters. DATASET_IRI is the dcat:Dataset IRI of the record to follow.
# Every URL the notebook fetches comes out of that record.
DATASET_IRI = "https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/22238845-1213-43f6-a60f-e84cdc9fa2ed"  # Huntingtons disease transcriptomics (GSE64810)
DOWNLOAD_DIR = "downloads"
OUTPUT_DIR = "output"

# How the downloaded file is laid out and which samples to compare. These describe the
# file's contents, not where it is. GSE64810 is one tab-separated matrix of DESeq2-normalised
# counts (gene x sample), and the sample names start with C_ (control) or H_ (Huntington).
DATA_SHEET = None                     # the sheet with the matrix, for a workbook
SAMPLE_SHEET = None                   # the sheet with one row per sample, if there is one
SAMPLE_ID_COLUMN = None
GROUP_COLUMN = None
GROUP_FROM_NAME = r"^([CH])_"         # regex whose first group is the label, when there is no sample sheet
CONTROL = r"^C$"                      # group labels matching this are the controls
CASE = "HD"                           # every other labelled sample
LOG_TRANSFORM = True                  # log2(x + 1) first, for count data

# EJPRD WP13 CAKUT mirnome instead: a workbook with a "Data matrix" sheet (miRNA x sample,
# RMA-normalised, already log scale) and a "Metadata" sheet, one row per sample.
# DATASET_IRI = "https://w3id.org/ejp-rd/fairdatapoints/wp13/dataset/0d2930d7-933d-4a5f-86e4-05e9537a18a0"
# DATA_SHEET, SAMPLE_SHEET = "Data matrix", "Metadata"
# SAMPLE_ID_COLUMN, GROUP_COLUMN, GROUP_FROM_NAME = "Sample name", "characteristics: group", None
# CONTROL, CASE, LOG_TRANSFORM = r"^Control", "CAKUT", False

assert DATASET_IRI, "Set DATASET_IRI to a dataset record IRI first."

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))   # the demonstrator folder, so that `src` imports
from src import firstlook, vpfollow

started = datetime.now(timezone.utc)

## 1. The dataset record

Fetch the record with content negotiation (an FDP serves Turtle) and read what it says about itself.

In [ ]:
dataset, distributions = vpfollow.follow(DATASET_IRI)

print("Title      :", dataset.title)
print("Publisher  :", dataset.publisher_name or dataset.publisher_iri)
print("Licence    :", dataset.license)
print("Access     :", dataset.access_rights)
print("Themes     :", ", ".join(dataset.themes) or "(none)")
print("Keywords   :", ", ".join(dataset.keywords) or "(none)")
print("Landing    :", dataset.landing_page)
print("Distributions:", len(distributions))
print()
print(dataset.description)

## 2. The distributions

Each distribution is its own record. We follow the one we can download.

In [ ]:
pd.DataFrame([{
    "title": d.title,
    "media type": d.media_type,
    "download URL": d.download_url,
    "access URL": d.access_url,
    "licence": d.license,
} for d in distributions])

In [ ]:
chosen = vpfollow.choose(distributions)
assert chosen is not None, "No distribution carries a download or access URL; see the README fallback."
print("Following:", chosen.title)
print("URL      :", chosen.data_url)

downloaded = vpfollow.download(chosen.data_url, DOWNLOAD_DIR)
print("Saved to :", downloaded, f"({downloaded.stat().st_size:,} bytes)")

## 3. The data

Load the file and look at it. A workbook says what is inside it, so list its sheets first. The analysis below is deliberately simple, a case-versus-control difference of means per feature. The point is that we got here from the metadata alone.

In [ ]:
print("Sheets:", vpfollow.sheets(downloaded) or "(text file)")
table = vpfollow.load_table(downloaded, sheet=DATA_SHEET)
matrix = firstlook.as_matrix(table)
if LOG_TRANSFORM:
    matrix = np.log2(matrix + 1)
print(f"{matrix.shape[0]:,} features x {matrix.shape[1]} samples")
matrix.iloc[:5, :6]

In [ ]:
if SAMPLE_SHEET:
    sample_sheet = vpfollow.load_table(downloaded, sheet=SAMPLE_SHEET)
    labels = firstlook.groups_from_sheet(sample_sheet, SAMPLE_ID_COLUMN, GROUP_COLUMN)
else:
    labels = firstlook.groups_from_names(matrix.columns, GROUP_FROM_NAME)
groups = firstlook.two_groups(labels, CONTROL, CASE)

print(labels.value_counts().to_string(), end="\n\n")
print(groups.value_counts().to_string())

In [ ]:
summary = firstlook.compare(matrix, groups)
case = summary.attrs["case"]
print("Samples compared:", summary.attrs["n"])
summary.head(20)

In [ ]:
top = summary.head(20)
fig, ax = plt.subplots(figsize=(8, 5))
top["difference"].plot.barh(ax=ax, color=np.where(top["difference"] > 0, "#c0504d", "#4f81bd"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"mean difference ({case} − control), log2 scale")
ax.set_ylabel("")
ax.set_title(dataset.title or DATASET_IRI)
ax.invert_yaxis()
plt.tight_layout()

Path(OUTPUT_DIR).mkdir(exist_ok=True)
figure_path = Path(OUTPUT_DIR) / "top_differences.png"
fig.savefig(figure_path, dpi=150)

## 4. Provenance

A small PROV-O record of what this run fetched and produced, so the analysis itself is traceable.

In [ ]:
prov = vpfollow.prov_graph(dataset, chosen, downloaded, started)
prov_path = Path(OUTPUT_DIR) / "prov.ttl"
prov.serialize(destination=str(prov_path), format="turtle")
print(prov_path.read_text())

## What this shows

- A dataset in the VP can be reached and used from its metadata alone, with the licence and publisher the record states known before any data is downloaded.
- The record is only as good as what was entered. The Huntington record names NCBI as publisher under an IRI that is a file path left by the script that populated the FDP, and calls a gzipped tab-separated file `text/csv`. For the WP13 CAKUT datasets the record's licence (CC BY-NC-ND 3.0) differs from the repository's (CC BY 4.0 on Zenodo). See `records.md`.
- The portal's resource list does not yet link to the record, so a user has to know the IRI. Closing that gap is a portal change worth requesting.
- The notebook itself can be registered as a resource in the demo index with FDPlight, which is the second act of this demonstrator.